In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1: Loading Libraries

In [107]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')  
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 2 : Reading the data #  

In [ ]:
df = pd.read_csv("/kaggle/input/market-segmentation-in-insurance-unsupervised/Customer Data.csv")
df

In [ ]:
# Dropping CUST_ID as it is not a numerical feature for clustering
df_cleaned = df.drop('CUST_ID', axis=1)

# Display the first few rows to understand the structure
df_cleaned.head()

In [ ]:
# Selecting Annual Income and Spending Score for clustering
X = df.iloc[:, [3, 4]].values

In [ ]:
df.columns

In [ ]:
df.describe().T

In [ ]:
# Selecting relevant features for clustering
# We focus on Balance, Purchases, Credit Limit, and Minimum Payments
features = ['BALANCE', 'PURCHASES', 'CREDIT_LIMIT', 'MINIMUM_PAYMENTS']
X_data = df[features]

# 3: Data Cleaning (Handling Missing Values)

In [ ]:
# Filling missing values with Median
df_cleaned['MINIMUM_PAYMENTS'].fillna(df_cleaned['MINIMUM_PAYMENTS'].median(), inplace=True)
df_cleaned['CREDIT_LIMIT'].fillna(df_cleaned['CREDIT_LIMIT'].median(), inplace=True)

# Final check for nulls
print(f"Missing values: {df_cleaned.isnull().sum().sum()}")

# 4: Handling Outliers

In [ ]:
# Apply log transformation to all numerical columns to handle outliers
# Using log1p to avoid errors with zero values
df_log = df_cleaned.apply(np.log1p)

# We will use all features for a comprehensive segmentation
X = df_log.values

In [ ]:
# Visualizing Outliers using Boxplot
plt.figure(figsize=(10, 5))
sns.boxplot(data=df[['BALANCE', 'PURCHASES', 'CREDIT_LIMIT']])
plt.title('Outliers Detection')
plt.show()


# 5 : Feature Scaling 

In [ ]:
# Scaling the features to ensure the algorithm treats them equally
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
num = df.drop("CUST_ID" , axis=1).columns
num

# 6: The Elbow Method 

In [ ]:
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

# Plotting the results
plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker='o', color='blue')
plt.title('The Elbow Method')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.grid(True)
plt.show()

# 5: Training the K-Means Model 

In [ ]:
# Training the model (Assume K=5)
kmeans = KMeans(n_clusters=5, init='k-means++', random_state=42)
clusters = kmeans.fit_predict(X_scaled)

# Add results to the dataframe
df['Cluster'] = clusters

# 6 : Visualizing the Cluster

In [ ]:
# Visualization
plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_scaled[:, 0], y=X_scaled[:, 2], hue=clusters, palette='viridis', s=50)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 2], s=200, c='red', marker='X', label='Centroids')
plt.title('Customer Segments Visualization')
plt.xlabel('Balance (Scaled)')
plt.ylabel('Purchases (Scaled)')
plt.legend()
plt.show()

# Conclusion:

# K-Means clustering (after data cleaning and feature scalling) successfully segmented ~9000 credit card customers into 5 distinct groups based on behavior.Key segments:High-activity / high-spend customers (frequent purchases, high credit limit usage).Low-activity / dormant customers (minimal transactions, low balance).High-balance / revolving users (carry high debt, frequent cash advances, minimum payments).Installment-focused customers.Balanced / moderate users.This segmentation enables targeted marketing, personalized offers, risk management, and improved customer retention strategies for the credit card company. Elbow method and silhouette analysis confirmed 5 as optimal clusters.
